In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/label-encoderv4/label_encoder.pkl
/kaggle/input/ocean-hazardv4/ocean_hazard.csv


In [2]:
from transformers import AutoTokenizer
from datasets import Dataset
import pandas as pd
import pickle

# Load your processed CSV dataset
df = pd.read_csv("/kaggle/input/ocean-hazardv4/ocean_hazard.csv")

# Load label encoder
with open("/kaggle/input/label-encoderv4/label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)

# Encode labels to numeric IDs
df['label_id'] = label_encoder.transform(df['label'])

# Convert to HuggingFace Dataset
hf_dataset = Dataset.from_pandas(df[['description', 'label_id']])

# Load tokenizer
model_name = "ai4bharat/indic-bert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize dataset
def tokenize(batch):
    texts = [str(x) for x in batch['description']]
    return tokenizer(texts, padding="max_length", truncation=True, max_length=128)

tokenized_dataset = hf_dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.rename_column("label_id", "labels")
tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])


config.json:   0%|          | 0.00/507 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/5.65M [00:00<?, ?B/s]

Map:   0%|          | 0/3361 [00:00<?, ? examples/s]

In [3]:
from datasets import DatasetDict

# Split 80/20 for train/test
split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split_dataset['train']
test_dataset = split_dataset['test']


In [4]:
from transformers import AutoModelForSequenceClassification

num_labels = len(label_encoder.classes_)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)


2025-09-12 06:53:45.325315: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757660025.662887      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757660025.759043      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


pytorch_model.bin:   0%|          | 0.00/135M [00:00<?, ?B/s]

Some weights of AlbertForSequenceClassification were not initialized from the model checkpoint at ai4bharat/indic-bert and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [5]:
pip cache purge

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


model.safetensors:   0%|          | 0.00/135M [00:00<?, ?B/s]

Files removed: 0
Note: you may need to restart the kernel to use updated packages.


In [6]:
import transformers
print(transformers.__version__)

4.52.4


In [7]:
!pip install evaluate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.5.1
    Uninstalling fsspec-2025.5.1:
      Successfully uninstalled fsspec-2025.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidi

In [8]:
from transformers import TrainingArguments
help(TrainingArguments.__init__)

Help on function __init__ in module transformers.training_args:

__init__(self, output_dir: Optional[str] = None, overwrite_output_dir: bool = False, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, per_gpu_train_batch_size: Optional[int] = None, per_gpu_eval_batch_size: Optional[int] = None, gradient_accumulation_steps: int = 1, eval_accumulation_steps: Optional[int] = None, eval_delay: Optional[float] = 0, torch_empty_cache_steps: Optional[int] = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: Union[transformers.trainer_utils.SchedulerType, str] = 'linear', lr_scheduler_kwargs: Unio

In [9]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from transformers import TrainingArguments, Trainer, TrainerCallback
from torch.nn import CrossEntropyLoss
from sklearn.metrics import accuracy_score
import numpy as np

# Define accuracy metric using sklearn
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

class SaveEvery100EpochsCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        if int(state.epoch) % 100 == 0:  # Every 100 epochs
            # Create custom save directory
            save_dir = f"./model_epoch_{int(state.epoch)}"
            
            # Create directory if it doesn't exist
            os.makedirs(save_dir, exist_ok=True)
            
            # Save model and tokenizer
            kwargs['model'].save_pretrained(save_dir)
            kwargs['tokenizer'].save_pretrained(save_dir)
            
            print(f"Model saved at epoch {int(state.epoch)} to {save_dir}")
            
            # Print current performance
            if hasattr(state, 'log_history') and state.log_history:
                latest_log = state.log_history[-1]
                if 'eval_accuracy' in latest_log:
                    print(f"Current accuracy: {latest_log['eval_accuracy']:.4f}")

training_args = TrainingArguments(
    output_dir="./models_500epochs",
    num_train_epochs=500,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",              
    save_strategy="epoch",                 
    logging_strategy="steps",           
    logging_steps=50,                   
    learning_rate=2e-5,
    weight_decay=0.01,
    save_total_limit=2,  
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True, 
    report_to=[]  
)

trainer = Trainer(
    model=model,                       
    args=training_args,
    train_dataset=train_dataset,       
    eval_dataset=test_dataset,
    tokenizer=tokenizer,                
    compute_metrics=compute_metrics,
    callbacks=[SaveEvery100EpochsCallback()]  
)

/tmp/ipykernel_36/3381309027.py:54: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:

trainer.train()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,0.910500,0.653471,0.741456
2,0.539500,0.548127,0.769688
3,0.435000,0.519144,0.777117
4,0.382000,0.534327,0.781575
5,0.312300,0.584825,0.780089
6,0.244700,0.596564,0.780089
7,0.227100,0.658620,0.790490
8,0.184200,0.688589,0.800892
9,0.168700,0.771153,0.796434
10,0.154600,0.899803,0.772660


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked t

KeyError: 'tokenizer'